In [ ]:
# =============================================================
# TAHAP 1: MEMBANGUN CASE BASE
# Mata Kuliah Penalaran Komputer - CBR Korupsi Kerugian Keuangan Negara
# =============================================================
# Jalankan file ini sebagai script Python atau salin ke Jupyter Notebook
# Setiap bagian diberi tanda # %%  agar bisa dijalankan per cell di Jupyter

# Tahap 1: Membangun Case Base
**Tujuan:** Mengumpulkan (scraping) dan menyiapkan (cleaning) corpus putusan
dari Direktori Putusan MA RI — Korupsi Kerugian Keuangan Negara

In [ ]:
Import Library
import os
import re
import time
import logging
import requests
from bs4 import BeautifulSoup
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Konfigurasi Path & Logging
BASE_DIR   = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) \
             if "__file__" in dir() else os.getcwd()
RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")
LOG_DIR    = os.path.join(BASE_DIR, "logs")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(LOG_DIR, "cleaning.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)
print("✅ Konfigurasi path & logging selesai")
print(f"   RAW_DIR : {RAW_DIR}")
print(f"   LOG_DIR : {LOG_DIR}")

In [ ]:
Konfigurasi Scraping
BASE_URL   = "https://putusan3.mahkamahagung.go.id"
KATEGORI   = "korupsi-kerugian-keuangan-negara-1"
LIST_URL   = f"{BASE_URL}/direktori/index/kategori/{KATEGORI}.html"
TARGET_DOC = 35          # unduh sedikit lebih dari 30 sebagai buffer
DELAY_SEC  = 2           # jeda antar request (etis)
HEADERS    = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

In [ ]:
Fungsi: Ambil daftar URL putusan dari halaman listing
def get_putusan_urls(max_docs: int = TARGET_DOC) -> list:
    """Scraping halaman listing untuk mendapatkan URL detail setiap putusan."""
    urls   = []
    page   = 1
    session = requests.Session()
    session.headers.update(HEADERS)

    while len(urls) < max_docs:
        list_page_url = (
            f"{BASE_URL}/direktori/index/kategori/{KATEGORI}/page/{page}.html"
        )
        print(f"  📄 Scraping halaman listing ke-{page}: {list_page_url}")
        try:
            resp = session.get(list_page_url, timeout=20)
            if resp.status_code != 200:
                print(f"  ⚠️  Status {resp.status_code}, berhenti.")
                break
            soup = BeautifulSoup(resp.text, "html.parser")
            # Link detail putusan biasanya berada di tag <a> dengan href /direktori/putusan/...
            links = soup.select("a[href*='/direktori/putusan/']")
            if not links:
                print("  ⚠️  Tidak ada link putusan ditemukan di halaman ini.")
                break
            for a in links:
                href = a.get("href", "")
                full = href if href.startswith("http") else BASE_URL + href
                if full not in urls:
                    urls.append(full)
                if len(urls) >= max_docs:
                    break
            print(f"  ✅ Total URL terkumpul: {len(urls)}")
            page += 1
            time.sleep(DELAY_SEC)
        except requests.exceptions.RequestException as e:
            print(f"  ❌ Error saat scraping halaman {page}: {e}")
            break

    return urls[:max_docs]

In [ ]:
Fungsi: Unduh PDF dari halaman detail putusan
def download_pdf(detail_url: str, save_path: str, session: requests.Session) -> bool:
    """Mengunjungi halaman detail putusan, lalu mengunduh file PDF-nya."""
    try:
        resp = session.get(detail_url, timeout=20)
        if resp.status_code != 200:
            return False
        soup = BeautifulSoup(resp.text, "html.parser")
        # Cari link download PDF
        pdf_link = None
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if href.lower().endswith(".pdf") or "download" in href.lower():
                pdf_link = href if href.startswith("http") else BASE_URL + href
                break
        if not pdf_link:
            logger.warning(f"PDF tidak ditemukan di {detail_url}")
            return False
        pdf_resp = session.get(pdf_link, timeout=60, stream=True)
        if pdf_resp.status_code == 200:
            with open(save_path, "wb") as f:
                for chunk in pdf_resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        return False
    except Exception as e:
        logger.error(f"Gagal unduh PDF dari {detail_url}: {e}")
        return False

In [ ]:
Fungsi: Ekstrak teks dari PDF
def extract_text_from_pdf(pdf_path: str) -> str:
    """Ekstrak plain text dari file PDF menggunakan pdfminer."""
    try:
        text = extract_text(pdf_path)
        return text if text else ""
    except (PDFSyntaxError, Exception) as e:
        logger.error(f"Gagal ekstrak teks dari {pdf_path}: {e}")
        return ""

In [ ]:
Fungsi: Bersihkan teks putusan
def clean_text(raw_text: str) -> str:
    """
    Membersihkan teks hasil ekstraksi PDF:
    - Hapus header/footer (nomor halaman, watermark, dll.)
    - Normalisasi spasi dan karakter
    - Lowercase
    """
    # Hapus karakter non-ASCII & karakter kontrol
    text = re.sub(r"[^\x20-\x7E\n\t]", " ", raw_text)
    # Hapus pola nomor halaman: "- 1 -", "Halaman 1 dari 20", "hal. 1"
    text = re.sub(r"-\s*\d+\s*-", " ", text)
    text = re.sub(r"[Hh]alaman\s*\d+\s*(dari\s*\d+)?", " ", text)
    text = re.sub(r"[Hh]al\.\s*\d+", " ", text)
    # Hapus watermark umum MA
    text = re.sub(r"mahkamah\s*agung\s*republik\s*indonesia", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"direktori\s*putusan", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"disclaimer\s*kepaniteraan.*", " ", text, flags=re.IGNORECASE | re.DOTALL)
    # Hapus baris kosong berlebihan
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Normalisasi spasi
    text = re.sub(r"[ \t]+", " ", text)
    # Lowercase
    text = text.lower().strip()
    return text

In [ ]:
Fungsi: Validasi teks (minimal 80% dari panjang rata-rata)
def validate_text(text: str, min_words: int = 200) -> bool:
    """Cek keutuhan teks: minimal sejumlah kata."""
    word_count = len(text.split())
    return word_count >= min_words

In [ ]:
Pipeline Utama: Scraping → Download → Clean → Simpan
def run_case_base_pipeline():
    print("\n" + "="*60)
    print("  TAHAP 1: MEMBANGUN CASE BASE")
    print("="*60)

    session = requests.Session()
    session.headers.update(HEADERS)

    # (a) Ambil daftar URL putusan
    print("\n[1/4] Mengambil daftar URL putusan...")
    putusan_urls = get_putusan_urls(max_docs=TARGET_DOC)
    print(f"  → {len(putusan_urls)} URL berhasil dikumpulkan")
    logger.info(f"Total URL dikumpulkan: {len(putusan_urls)}")

    # (b) Unduh PDF, ekstrak & bersihkan teks
    print("\n[2/4] Mengunduh PDF, mengekstrak & membersihkan teks...")
    berhasil, gagal, invalid = 0, 0, 0

    for idx, url in enumerate(putusan_urls, start=1):
        case_id    = f"case_{idx:03d}"
        pdf_path   = os.path.join(RAW_DIR, f"{case_id}.pdf")
        txt_path   = os.path.join(RAW_DIR, f"{case_id}.txt")

        # Skip jika sudah ada
        if os.path.exists(txt_path):
            print(f"  [{idx}/{len(putusan_urls)}] {case_id} sudah ada, skip.")
            berhasil += 1
            continue

        print(f"  [{idx}/{len(putusan_urls)}] Memproses {case_id}...")

        # Download PDF
        ok = download_pdf(url, pdf_path, session)
        if not ok:
            print(f"    ❌ Gagal unduh PDF.")
            logger.warning(f"{case_id}: Gagal unduh dari {url}")
            gagal += 1
            time.sleep(DELAY_SEC)
            continue

        # Ekstrak teks
        raw_text = extract_text_from_pdf(pdf_path)
        if not raw_text:
            print(f"    ⚠️  Gagal ekstrak teks.")
            logger.warning(f"{case_id}: Gagal ekstrak teks")
            gagal += 1
            time.sleep(DELAY_SEC)
            continue

        # Bersihkan teks
        clean = clean_text(raw_text)

        # Validasi
        if not validate_text(clean):
            print(f"    ⚠️  Teks terlalu pendek (tidak valid).")
            logger.warning(f"{case_id}: Teks tidak valid (< 200 kata)")
            invalid += 1
            time.sleep(DELAY_SEC)
            continue

        # Simpan .txt
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(clean)

        # Hapus PDF setelah teks disimpan (hemat storage)
        if os.path.exists(pdf_path):
            os.remove(pdf_path)

        logger.info(f"{case_id}: OK — {len(clean.split())} kata")
        print(f"    ✅ Tersimpan: {txt_path} ({len(clean.split())} kata)")
        berhasil += 1
        time.sleep(DELAY_SEC)

    # (c) Laporan akhir
    print("\n[3/4] Validasi hasil...")
    txt_files = [f for f in os.listdir(RAW_DIR) if f.endswith(".txt")]
    print(f"  Total file .txt tersimpan: {len(txt_files)}")
    logger.info(f"Pipeline selesai | Berhasil: {berhasil} | Gagal: {gagal} | Invalid: {invalid}")

    print("\n[4/4] Ringkasan:")
    print(f"  ✅ Berhasil : {berhasil}")
    print(f"  ❌ Gagal    : {gagal}")
    print(f"  ⚠️  Invalid  : {invalid}")
    print(f"  📁 File .txt: {len(txt_files)}")
    print(f"\n  Log tersimpan di: {os.path.join(LOG_DIR, 'cleaning.log')}")
    print("="*60)

    if len(txt_files) < 30:
        print(f"\n⚠️  PERINGATAN: Hanya {len(txt_files)} dokumen valid.")
        print("   Tambahkan lebih banyak data agar memenuhi syarat ≥ 30 dokumen.\n")
    else:
        print(f"\n✅ {len(txt_files)} dokumen siap digunakan untuk tahap selanjutnya.\n")

if __name__ == "__main__":
    run_case_base_pipeline()